In [2]:
from datasets import load_dataset
from LLMGeometry.datasets import CACHE_DIR
import numpy as np
import pandas as pd

In [ ]:
raw_dataset = load_dataset("trec", trust_remote_code=True, cache_dir=CACHE_DIR) # Loading TREC dataset

In [4]:
test_df = raw_dataset['test'].to_pandas()
train_df = raw_dataset['train'].to_pandas()


def process_dataset(df, random_state=42):
    df = df.copy()

    # Removing rows with coarse_label = 0
    df = df[df['coarse_label'] != 0]

    # For remaining labels subselecting so that they are equally represented (to avoid class imbalance). Find the smallest class size and subselect all classes to that size
    min_class_size = df['coarse_label'].value_counts().min()
    df = df.groupby('coarse_label').apply(lambda x: x.sample(min_class_size, random_state=random_state)).reset_index(drop=True)

    # Renaming columns
    df = df.rename(columns={'coarse_label': 'label', 'text': 'text'})
    df = df.drop(columns=['fine_label'])

    # Mapping labels
    TREC_mapping = {
        1: 'entity',
        2: 'description',
        3: 'human',
        4: 'location',
        5: 'numeric'
    }

    letter_mapping = {
        1: 'A',
        2: 'B',
        3: 'C',
        4: 'D',
        5: 'E',
    }
    
    df['category'] = df['label'].map(TREC_mapping)
    df['category'] = df['category'].str.title()
    df['category_letter'] = df['label'].map(letter_mapping) 

    # Shuffling
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return df

In [5]:
test_df = process_dataset(test_df)
train_df = process_dataset(train_df)

In [6]:
test_df

,text,label,category,category_letter
0,What strait separates North America from Asia ?,4,Location,D
1,What is supernova ?,2,Description,B
2,Where is the volcano Mauna Loa ?,4,Location,D
3,What do you call a professional map drawer ?,1,Entity,A
4,What is amitriptyline ?,2,Description,B
...,...,...,...,...
320,Who was the first African American to play for...,3,Human,C
321,What is home equity ?,2,Description,B
322,What is cryogenics ?,2,Description,B
323,When is Father 's Day ?,5,Numeric,E


In [7]:
category_shuffle_mapping = {
    'Entity' : 'Location',
    'Location' : 'Human',
    'Human' : 'Description',
    'Description' : 'Numeric',
    'Numeric' : 'Entity',
}

ds['train']['category_shuffled'] = ds['train']['category'].map(category_shuffle_mapping)
ds['test']['category_shuffled'] = ds['test']['category'].map(category_shuffle_mapping)


In [11]:
ds['train']

,text,label,category,category_letter,category_shuffled
0,Which of the following TV newsmen was a Rhodes...,2,Human,C,Description
1,Who portrayed Vincent Van Gogh in Lust for Life ?,2,Human,C,Description
2,Who received the Will Rogers Award in 1989 ?,2,Human,C,Description
3,Who 's the lead singer of the Led Zeppelin band ?,2,Human,C,Description
4,How many millimeters are in a mile ?,4,Numeric,E,Entity
...,...,...,...,...,...
4170,How many verses are in the Bible ?,4,Numeric,E,Entity
4171,What turns blue litmus paper red ?,0,Entity,A,Location
4172,What is the country of origin for the name Tho...,3,Location,D,Human
4173,"What day is August 13 , 1971 ?",4,Numeric,E,Entity


In [21]:
with open('TREC_coarse.pickle', 'wb') as f:
    pd.to_pickle({'train': train_df, 'test': test_df}, f)